# Fake Job Posting Detection

## 1. Project Overview and Data Loading
This notebook documents the end-to-end pipeline for detecting fraudulent job postings. The dataset exhibits a severe class imbalance, which dictates our evaluation metrics and preprocessing strategy.

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Load the raw dataset
df = pd.read_csv('../data/raw/fake_job_postings.csv')
print(f"Dataset shape: {df.shape}")
print(f"Target distribution:\n{df['fraudulent'].value_counts(normalize=True)}")

Dataset shape: (17880, 18)
Target distribution:
fraudulent
0    0.951566
1    0.048434
Name: proportion, dtype: float64


## 2. Exploratory Data Analysis
Before modeling, we analyzed missing values, text length distributions, and categorical fraud rates to inform our imputation and feature engineering strategies. We also explored initial word frequencies to identify potential fraud triggers.

* Text columns with missing values will be filled with empty strings.
* Categorical columns with missing values will be explicitly assigned an 'Unknown' category.

In [6]:
# Missing Value Analysis
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df)) * 100
print("Top Missing Value Percentages:")
print(missing_pct[missing_pct > 0].head())

# Categorical Fraud Rate Example
print("\nFraud Rate by Employment Type:")
print(df.groupby('employment_type')['fraudulent'].mean().sort_values(ascending=False))

Top Missing Value Percentages:
salary_range           83.959732
department             64.580537
required_education     45.329978
benefits               40.335570
required_experience    39.429530
dtype: float64

Fraud Rate by Employment Type:
employment_type
Part-time    0.092848
Other        0.066079
Full-time    0.042169
Contract     0.028871
Temporary    0.008299
Name: fraudulent, dtype: float64


## 3. Preprocessing Pipeline
The preprocessing stage transforms raw text and categorical variables into a sparse matrix. To prevent data leakage, all transformations are fitted exclusively on the training set.

**Methodology:**
1. **Text Cleaning:** Lowercasing, HTML tag removal, and whitespace normalization.
2. **Train/Test Split:** Stratified 70/30 split to preserve the 5% fraud distribution.
3. **TF-IDF Vectorization:** Capped at 5000 features with bigrams to control dimensionality.
4. **One-Hot Encoding:** Applied to 5 categorical features, dropping original text.
5. **Sparse Stacking:** Combined into a highly memory-efficient compressed sparse row (CSR) matrix.

In [7]:
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack

def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'<[^>]+>', '', text) 
    text = re.sub(r'[^a-z\s]', ' ', text) 
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Stratified Split
X = df.drop('fraudulent', axis=1)
y = df['fraudulent']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)

# Text Processing
text_cols = ['title', 'description', 'requirements', 'company_profile']
X_train_text = X_train[text_cols].fillna('').agg(' '.join, axis=1).apply(clean_text)
X_test_text = X_test[text_cols].fillna('').agg(' '.join, axis=1).apply(clean_text)

# TF-IDF
vectorizer = TfidfVectorizer(max_features=5000, min_df=5, max_df=0.7, ngram_range=(1, 2), stop_words='english', dtype=np.float32)
X_train_tfidf = vectorizer.fit_transform(X_train_text)
X_test_tfidf = vectorizer.transform(X_test_text)

# Categorical Encoding
cat_cols = ['location', 'department', 'employment_type', 'required_experience', 'industry']
cat_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True, dtype=np.float32)
X_train_cat = cat_encoder.fit_transform(X_train[cat_cols].fillna('Unknown'))
X_test_cat = cat_encoder.transform(X_test[cat_cols].fillna('Unknown'))

# Final Sparse Matrix
X_train_processed = hstack([X_train_tfidf, X_train_cat]).tocsr()
X_test_processed = hstack([X_test_tfidf, X_test_cat]).tocsr()
train_sparsity = 100 * (1.0 - X_train_processed.nnz / (X_train_processed.shape[0] * X_train_processed.shape[1]))

print(f"Final Train Shape: {X_train_processed.shape}")
print(f"Final Test Shape:  {X_test_processed.shape}")
print(f"Matrix Sparsity:   {train_sparsity:.2f}%")

Final Train Shape: (12516, 8770)
Final Test Shape:  (5364, 8770)
Matrix Sparsity:   98.16%
